In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np

In [5]:
from bayesgpt.simulators import NestedModelFamily
from bayesgpt.simulators.benchmarks import StandardDDM, CollapsingBoundDDM, SuperDDM

In [6]:
# Define global parameter space (superset)
param_names = [
    "v",  # drift
    "a",  # boundary
    "z",  # initial bias
    "tau",  # non-decision time
    "sigma",  # noise scale
    "angle",  # collapse rate
    "s_v",  # All 's' are variability
    "s_a",
    "s_z",
    "s_tau",
    "s_sigma",
    "s_angle",
]

In [7]:
model_family = NestedModelFamily(parameter_names=param_names)

In [8]:
rng = np.random.default_rng(seed=42)

free_params = {
    "v": lambda n: rng.normal(loc=0.5, scale=0.2, size=n),
    "a": lambda n: rng.uniform(low=0.6, high=1.4, size=n),
    "z": lambda n: rng.beta(a=2.0, b=2.0, size=n),
    "tau": lambda n: rng.uniform(low=0.1, high=0.9, size=n),
    "sigma": lambda n: np.full(n, 1.0),
    "s_v": lambda n: rng.gamma(shape=0.5, scale=1.0, size=n),
}

fixed_params = {
    "angle": 0.0,
    "s_a": 0.0,
    "s_z": 0.0,
    "s_tau": 0.0,
    "s_sigma": 0.0,
    "s_angle": 0.0,
    "v_components": np.array([-0.2, 0.0, 0.6], dtype=float),
    "p_components": np.array([0.2, 0.3, 0.5], dtype=float),
}

In [9]:
model_family.add_variant(
    name="super_ddm",
    model=SuperDDM,
    free_parameters=free_params,
    fixed_parameters=fixed_params,
    fallback_value=0.0,
)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (1, 14) + inhomogeneous part.

In [9]:
samples = model_family.sample("super_ddm", batch_size=100)

In [10]:
samples

{'sim_data': array([[0.8890028 , 0.        ],
        [0.74198683, 1.        ],
        [0.93819429, 1.        ],
        [1.26309582, 1.        ],
        [1.02475757, 1.        ],
        [0.9278745 , 1.        ],
        [0.87075979, 1.        ],
        [1.31143214, 0.        ],
        [0.52897876, 1.        ],
        [2.67733358, 1.        ],
        [1.4982608 , 1.        ],
        [1.14994572, 1.        ],
        [0.16985542, 0.        ],
        [0.78836937, 0.        ],
        [2.87464783, 1.        ],
        [1.67176133, 1.        ],
        [3.63406647, 1.        ],
        [0.48557658, 0.        ],
        [0.71416187, 0.        ],
        [0.83195878, 0.        ],
        [0.37042583, 1.        ],
        [1.7802445 , 1.        ],
        [0.86560656, 1.        ],
        [1.89797655, 1.        ],
        [1.36736524, 1.        ],
        [1.20954517, 0.        ],
        [0.76269727, 1.        ],
        [0.40217097, 1.        ],
        [0.59762812, 1.        ],
  

In [11]:
model_family.add_variant(
    name="ddm",
    model=StandardDDM,
    free_parameters={"v": lambda n: np.random.normal(1.0, 0.2, size=n)},
    fixed_parameters={"a": 1.0, "z": 0.5, "tau": 0.3, "s_v": 0.1, "sigma": 1.0},
)

In [12]:
model_family.add_variant(
    name="collapsing",
    model=CollapsingBoundDDM,
    free_parameters={
        "v": lambda n: np.random.normal(1.0, 0.2, size=n),
        "angle": lambda n: np.random.uniform(0.0, 0.5, size=n),
    },
    fixed_parameters={"a": 1.0, "z": 0.5, "tau": 0.3, "s_v": 0.1, "sigma": 1.0},
)

In [17]:
out = model_family.sample("ddm", batch_size=100)
mask = model_family.get_infer_mask("ddm", batch_size=100)
condition = model_family.get_variant_encoder("ddm", batch_size=100)

In [18]:
print(out["sim_data"].shape)
print(out["full_params"].shape)
print(mask.shape)
print(condition.shape)

(100, 2)
(100, 12)
(100, 12)
(100, 3)


In [29]:
out["sim_data"]

array([[0.575, 1.   ],
       [0.428, 1.   ],
       [0.615, 1.   ],
       [0.558, 0.   ],
       [1.13 , 1.   ],
       [0.489, 0.   ],
       [0.401, 0.   ],
       [0.41 , 1.   ],
       [0.974, 1.   ],
       [0.484, 1.   ],
       [0.456, 1.   ],
       [0.633, 0.   ],
       [0.641, 1.   ],
       [0.816, 1.   ],
       [0.599, 1.   ],
       [0.355, 0.   ],
       [0.949, 0.   ],
       [0.625, 1.   ],
       [0.399, 1.   ],
       [0.4  , 1.   ],
       [0.494, 1.   ],
       [0.578, 1.   ],
       [0.558, 1.   ],
       [0.842, 1.   ],
       [0.532, 1.   ],
       [0.425, 1.   ],
       [0.382, 1.   ],
       [0.49 , 1.   ],
       [1.3  , 1.   ],
       [0.722, 0.   ],
       [0.751, 1.   ],
       [0.708, 1.   ],
       [0.601, 1.   ],
       [0.775, 0.   ],
       [0.445, 1.   ],
       [1.065, 1.   ],
       [0.586, 1.   ],
       [0.383, 1.   ],
       [0.745, 1.   ],
       [0.378, 1.   ],
       [0.375, 1.   ],
       [0.449, 0.   ],
       [1.404, 1.   ],
       [0.4

In [30]:
out["full_params"]

array([[0.554168  , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.96912664, 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.80194896, 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [1.355698  , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.5866667 , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [1.0332814 , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.95180297, 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.9996534 , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [1.2273102 , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.83825034, 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0. 